# R1 (research track): VGGT vs VGGT + BA vs COLMAP, scored by pose error **and** by the 3DGS twin

First milestone of the README's Phase 4 ([proposal](../docs/research/hybridsfm_proposal.md)). For each scene, the same **N evenly spaced images** are posed four ways, then the same short gsplat recipe is trained on each set of poses:

| Method | What it is | Runs in |
|---|---|---|
| `gt` | Official Mip-NeRF 360 poses (COLMAP on all ~200 images), cut down to the N images. The reference | — |
| `colmap` | COLMAP on the N images only: SIFT, exhaustive matching, incremental mapping, one PINHOLE camera | CPU |
| `vggt` | VGGT-1B, one forward pass (VGGT's `demo_colmap.py`) | GPU |
| `vggt_ba` | VGGT + bundle adjustment (`demo_colmap.py --use_ba`: VGGSfM tracks, pycolmap BA) | GPU + CPU |

**Outputs** (all in the version's **Output**):
- `r1_results.md`: paste this back into the chat. `r1_results.json` is the same data, machine-readable.
- `figures/`: one test view per method.
- `r1_models/`: the four COLMAP models per scene (small; reused in R2).

**Questions this answers:**
1. How many images fit in one VGGT pass on a 16 GB T4? This sets the chunk size for R2.
2. How far are VGGT and VGGT+BA from the reference poses (AUC@30, RRA/RTA@5, ATE)?
3. **Does that pose gap show up in the twin?** Compare the PSNR / LPIPS of 3DGS trained on each method's poses. Feed-forward vs classical has not been compared this way, and the routing paper builds on it.

**Protocol notes:**
- **One VGGT pass per scene.** N is capped by T4 memory; chunking is R2. The memory sweep runs first and lowers N if needed.
- **Pseudo-ground truth.** `gt` is COLMAP on *all* images, so it favours `colmap` a little (same algorithm family, a subset of the same images). We state this in the paper.
- **Training and test split.** gsplat holds out every 8th of the N images by name. Each method's poses for those test images come from its own reconstruction, the standard practice for pose-free novel-view synthesis.
- **Short recipe.** MCMC with a 300k cap, 7k steps. This is the label recipe planned for R2; set `STEPS = 30_000` for full quality.

**How to run (Kaggle):**
1. *Create → New Notebook → File → Import Notebook* → this file.
2. *Settings*: **Accelerator GPU T4 x2** (training is pinned to one GPU: F2.2) and **Internet on**.
3. *Save Version → Save & Run All*.

Expect about 2–2.5 h:
- installs: gsplat compile ~15 min, VGGT environment ~5 min, weights 5 GB;
- per scene: ~15 min of COLMAP on CPU, ~5 min of VGGT, ~20 min of 3DGS.

**Your own scene (optional):** upload a folder with `images/` and `sparse/0/` (your full COLMAP run, used as pseudo-ground truth) as a Kaggle Dataset. *Add Input*, then put its path in `OWN_SCENES`.

## 1. Settings

In [ ]:
import os, sys, glob, json, time, re, subprocess, shutil
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # Kaggle T4 x2: use one GPU (gsplat would otherwise go multi-GPU, F2.2)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout)

SCENES     = ['garden', 'bicycle']           # Mip-NeRF 360 (official poses = reference)
OWN_SCENES = {}                              # e.g. {'table': '/kaggle/input/table-scene'}: needs images/ + sparse/0
N_VIEWS    = 40                              # images per scene, one VGGT pass. Lowered automatically if it won't fit
SWEEP      = [8, 16, 24, 32, 40, 48, 64, 80] # VGGT memory sweep (stops at the first out-of-memory)
METHODS    = ['gt', 'colmap', 'vggt', 'vggt_ba']
STEPS      = 7_000                           # short 3DGS recipe (the R2 label recipe); 30_000 for full quality
CAP        = 300_000
GSPLAT_ARGS = ['mcmc', '--strategy.cap-max', str(CAP)]

PROJECT = '/kaggle/working'                  # becomes the Output
WORK    = '/kaggle/tmp'
OUT     = PROJECT                            # R1 tables, figures, models
FIG     = f'{OUT}/figures'
R1      = f'{WORK}/r1'                       # per-scene, per-method scene folders (images are symlinks)
for d in [f'{PROJECT}/results', f'{PROJECT}/cache', f'{OUT}/r1_models', WORK, f'{WORK}/hybrid_sfm', FIG, R1]:
    os.makedirs(d, exist_ok=True)
run_name = lambda scene, method: f'r1_{scene}_{method}_{N_EFF}v_{STEPS // 1000}k'

import socket
try:
    socket.create_connection(('pypi.org', 443), timeout=10).close(); print('internet: ok')
except OSError:
    raise SystemExit('❌ No internet. Kaggle: Settings → Internet → On (needs a phone-verified account). Then run again.')

## 2. Install gsplat (main environment)

Same as `02c`: compiles once (~15 min), and saves the compiled library to the Output.

In [ ]:
import importlib.util
MARKER = '/tmp/.gs_deps_ok'

if not os.path.exists(MARKER):
    gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gsplat==1.5.3'], check=True)

    REPO = f'{WORK}/gsplat'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v1.5.3',
                        'https://github.com/nerfstudio-project/gsplat.git', REPO], check=True)
    reqs = open(f'{REPO}/examples/requirements.txt').read()

    # F0.1 + F0.2: git deps (several glued onto one line) -> split, build without isolation
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'setuptools', 'wheel'], check=True)
    if 'T4' in gpu_name:
        os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'   # faster builds, T4 only
    for url in re.findall(r'git\+https://github\.com/[^@\s]+@[0-9a-f]{40}', reqs):
        print('building', url.split('/')[-1][:60])
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', url], check=True)

    # F0.3: drop git lines + numpy<2 pin, install the rest
    rest = [l for l in reqs.splitlines() if 'git+https' not in l and not l.lower().startswith('numpy')]
    open('/tmp/reqs.txt', 'w').write('\n'.join(rest))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/tmp/reqs.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'remotezip', 'plyfile'], check=True)

    # F0.5: pycolmap fork uses np.uint64(-1) -> OverflowError on numpy 2 (patch the file, don't import it)
    f = os.path.join(os.path.dirname(importlib.util.find_spec('pycolmap').origin), 'scene_manager.py')
    s = open(f).read()
    if 'np.uint64(-1)' in s:   # harmless on numpy 1, required on numpy 2
        open(f, 'w').write(s.replace('np.uint64(-1)', 'np.uint64(18446744073709551615)')); print('patched pycolmap')

    # F0.4: make gsplat's examples/datasets a regular package (beats HF `datasets`)
    open(f'{REPO}/examples/datasets/__init__.py', 'a').close()

    open(MARKER, 'w').write('ok')

REPO = f'{WORK}/gsplat'
check = subprocess.run([sys.executable, '-c',
    'import numpy, gsplat, fused_ssim, pycolmap, tyro; print("numpy", numpy.__version__, "| gsplat", gsplat.__version__)'],
    capture_output=True, text=True, cwd=f'{REPO}/examples')
print(check.stdout or check.stderr[-2000:])

In [ ]:
os.environ['GSPLAT_PREBUILT_DIR'] = '/kaggle/tmp/gsplat_prebuilt'
# F1.5 fix, Kaggle version: the compiled library is saved to the Output (cache/gsplat_jit/...).
# To skip the ~15 min compile next time: add that Output as a Kaggle Dataset and point JIT_CACHE at it.
import torch, importlib.util
PREBUILT = '/kaggle/tmp/gsplat_prebuilt'
cap = torch.cuda.get_device_capability()
KEY = f"gsplat1.5.3_torch{torch.__version__}_py{sys.version_info.major}{sys.version_info.minor}_sm{cap[0]}{cap[1]}"
JIT_CACHE = f'{PROJECT}/cache/gsplat_jit/{KEY}'

# 1. Patch gsplat's loader: use $GSPLAT_PREBUILT_DIR/gsplat_cuda.so if it exists, otherwise behave as before
BACKEND = os.path.join(os.path.dirname(importlib.util.find_spec('gsplat').origin), 'cuda', '_backend.py')
src = open(BACKEND).read()
if 'GSPLAT_PREBUILT_DIR' not in src:
    old = '    from gsplat import csrc as _C\n'
    new = ('    _pre = os.environ.get("GSPLAT_PREBUILT_DIR", "/kaggle/tmp/gsplat_prebuilt")\n'
           '    if os.path.exists(os.path.join(_pre, "gsplat_cuda.so")):\n'
           '        _C = _import_module_from_library("gsplat_cuda", _pre, True)\n'
           '    else:\n'
           '        from gsplat import csrc as _C\n')
    assert src.count(old) == 1, 'gsplat _backend.py changed, patch needs updating'
    open(BACKEND, 'w').write(src.replace(old, new)); print('patched gsplat loader')

# 2. Restore the cached library from Drive
os.makedirs(PREBUILT, exist_ok=True)
if not os.path.exists(f'{PREBUILT}/gsplat_cuda.so') and os.path.exists(f'{JIT_CACHE}/gsplat_cuda.so'):
    shutil.copy(f'{JIT_CACHE}/gsplat_cuda.so', PREBUILT)

def load_gsplat_cuda():
    t = time.time()
    r = subprocess.run([sys.executable, '-c', 'from gsplat.cuda._backend import _C; print("SO=", _C.__file__ if _C else None)'],
                       capture_output=True, text=True)
    so = re.search(r'SO= (.+)', r.stdout)
    return (so.group(1).strip() if so else None), time.time() - t, r

so, dt, r = load_gsplat_cuda()
if so and so.startswith(PREBUILT):
    print(f'✅ gsplat CUDA loaded from the cache in {dt:.0f}s ({KEY})')
else:
    # 3. No usable cache: the import above compiled it (~15 min). Save the result for next time.
    if not so or so == 'None':
        print(r.stdout[-1500:], r.stderr[-3000:]); raise RuntimeError('gsplat CUDA failed to load or compile')
    print(f'compiled gsplat CUDA in {dt/60:.1f} min -> saving to the Output')
    os.makedirs(JIT_CACHE, exist_ok=True)
    shutil.copy(so, f'{JIT_CACHE}/gsplat_cuda.so'); shutil.copy(so, f'{PREBUILT}/gsplat_cuda.so')
    so2, dt2, _ = load_gsplat_cuda()
    print(f"{'✅' if so2 and so2.startswith(PREBUILT) else '❌'} reload from cache: {dt2:.0f}s ({so2})")

## 3. VGGT environment (separate Python 3.12)

VGGT's `demo_colmap.py` needs the **official** `pycolmap==3.10.0`, `pyceres==2.3` and LightGlue. gsplat's examples install a *fork* also called `pycolmap`, and Kaggle runs Python 3.13, which has no `pycolmap 3.10` wheels. So VGGT and the COLMAP baseline get their own environment, built with `uv`, with VGGT's pinned `torch 2.3.1`. Everything there runs as a subprocess.

In [ ]:
VENV, VGGT_REPO = f'{WORK}/vggt_env', f'{WORK}/vggt'
VPY = f'{VENV}/bin/python'
def sh(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:]); raise RuntimeError(f'failed: {" ".join(map(str, cmd))[:200]}')
    return r.stdout

if not os.path.exists(f'{VENV}/.ok'):
    t = time.time()
    sh([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
    sh([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', VENV])
    UV = [sys.executable, '-m', 'uv', 'pip', 'install', '--python', VPY]
    sh(UV + ['torch==2.3.1', 'torchvision==0.18.1', '--extra-index-url', 'https://download.pytorch.org/whl/cu121'])
    sh(UV + ['numpy==1.26.4', 'Pillow', 'huggingface_hub', 'einops', 'safetensors', 'opencv-python-headless',
             'trimesh', 'scipy', 'tqdm', 'pycolmap==3.10.0', 'pyceres==2.3',
             'lightglue @ git+https://github.com/jytime/LightGlue.git'])
    if not os.path.exists(VGGT_REPO):
        sh(['git', 'clone', '-q', '--depth', '1', 'https://github.com/facebookresearch/vggt.git', VGGT_REPO])
    open(f'{VENV}/.ok', 'w').write('ok')
    print(f'VGGT environment ready in {(time.time() - t) / 60:.1f} min')

VGGT_COMMIT = sh(['git', '-C', VGGT_REPO, 'rev-parse', '--short', 'HEAD']).strip()
VENV_ENV = {**os.environ, 'PYTHONPATH': VGGT_REPO, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
if not os.path.exists(f'{VENV}/.demo_ok'):            # everything demo_colmap.py imports, minus the web-demo extras
    reqs = [l.strip() for f in ['requirements.txt', 'requirements_demo.txt'] if os.path.exists(f'{VGGT_REPO}/{f}')
            for l in open(f'{VGGT_REPO}/{f}') if l.strip() and not l.startswith('#')]
    reqs = [r for r in reqs if not re.match(r'(gradio|viser|onnxruntime|torch|torchvision|numpy|pycolmap|pyceres|lightglue)\b', r, re.I)]
    sh([sys.executable, '-m', 'uv', 'pip', 'install', '--python', VPY, *reqs,
        'numpy==1.26.4', 'pycolmap==3.10.0', 'pyceres==2.3', 'matplotlib'])
    open(f'{VENV}/.demo_ok', 'w').write('ok')
r = subprocess.run([VPY, '-c', 'import demo_colmap'], cwd=VGGT_REPO, env=VENV_ENV, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-3000:]); raise RuntimeError('demo_colmap.py cannot be imported: see the error above')
print(sh([VPY, '-c', 'import torch, pycolmap, pyceres, lightglue; '
          'print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| pycolmap", pycolmap.__version__)'],
         env=VENV_ENV).strip(), '| vggt', VGGT_COMMIT)

## 4. Research code

These files are copies of `hybrid_sfm/` in the repo, tested locally with `python hybrid_sfm/tests/test_r1.py`:
- COLMAP binary I/O and subsetting;
- pose metrics;
- the VGGT memory sweep;
- the COLMAP baseline.

In [ ]:
%%writefile /kaggle/tmp/hybrid_sfm/colmap_io.py
"""Minimal COLMAP model I/O (binary and text), numpy only.

Written for the research track so pose evaluation and subsetting don't depend on a particular pycolmap
version (gsplat's examples install a pycolmap *fork* that shadows the official package; VGGT's
demo_colmap.py needs official pycolmap 3.10).

Model = dict with
    cameras: {camera_id: dict(model, width, height, params)}
    images:  {image_id: dict(qvec (w,x,y,z), tvec, camera_id, name, xys (P,2), point3D_ids (P,))}
    points:  {point3D_id: dict(xyz, rgb, error, image_ids, point2D_idxs)}
Poses are world-to-camera (COLMAP / OpenCV convention): x_cam = R(q) @ x_world + t.
"""
import os
import struct

import numpy as np

CAMERA_MODELS = {  # id: (name, number of params)
    0: ("SIMPLE_PINHOLE", 3), 1: ("PINHOLE", 4), 2: ("SIMPLE_RADIAL", 4), 3: ("RADIAL", 5),
    4: ("OPENCV", 8), 5: ("OPENCV_FISHEYE", 8), 6: ("FULL_OPENCV", 12), 7: ("FOV", 5),
    8: ("SIMPLE_RADIAL_FISHEYE", 4), 9: ("RADIAL_FISHEYE", 5), 10: ("THIN_PRISM_FISHEYE", 12),
}
MODEL_ID = {name: i for i, (name, _) in CAMERA_MODELS.items()}


def _read(f, fmt):
    return struct.unpack("<" + fmt, f.read(struct.calcsize("<" + fmt)))


# ------------------------------------------------------------------ binary
def read_cameras_bin(path):
    cams = {}
    with open(path, "rb") as f:
        for _ in range(_read(f, "Q")[0]):
            cid, mid, w, h = _read(f, "iiQQ")
            name, n = CAMERA_MODELS[mid]
            cams[cid] = dict(model=name, width=w, height=h, params=np.array(_read(f, "d" * n)))
    return cams


def read_images_bin(path):
    imgs = {}
    with open(path, "rb") as f:
        for _ in range(_read(f, "Q")[0]):
            iid, = _read(f, "i")
            q = np.array(_read(f, "dddd"))
            t = np.array(_read(f, "ddd"))
            cid, = _read(f, "i")
            name = b""
            while (c := f.read(1)) != b"\x00":
                name += c
            n, = _read(f, "Q")
            raw = np.frombuffer(f.read(24 * n), dtype=np.dtype([("x", "<f8"), ("y", "<f8"), ("id", "<i8")]))
            imgs[iid] = dict(qvec=q, tvec=t, camera_id=cid, name=name.decode(),
                             xys=np.stack([raw["x"], raw["y"]], 1), point3D_ids=raw["id"].copy())
    return imgs


def read_points_bin(path):
    pts = {}
    with open(path, "rb") as f:
        for _ in range(_read(f, "Q")[0]):
            pid, = _read(f, "Q")
            xyz = np.array(_read(f, "ddd"))
            rgb = np.array(_read(f, "BBB"), np.uint8)
            err, n = _read(f, "dQ")
            tr = np.frombuffer(f.read(8 * n), dtype="<i4").reshape(-1, 2)
            pts[pid] = dict(xyz=xyz, rgb=rgb, error=err, image_ids=tr[:, 0].copy(), point2D_idxs=tr[:, 1].copy())
    return pts


def write_model_bin(model, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "cameras.bin"), "wb") as f:
        f.write(struct.pack("<Q", len(model["cameras"])))
        for cid, c in sorted(model["cameras"].items()):
            f.write(struct.pack("<iiQQ", cid, MODEL_ID[c["model"]], int(c["width"]), int(c["height"])))
            f.write(struct.pack("<" + "d" * len(c["params"]), *map(float, c["params"])))
    with open(os.path.join(out_dir, "images.bin"), "wb") as f:
        f.write(struct.pack("<Q", len(model["images"])))
        for iid, im in sorted(model["images"].items()):
            f.write(struct.pack("<i4d3di", iid, *map(float, im["qvec"]), *map(float, im["tvec"]), im["camera_id"]))
            f.write(im["name"].encode() + b"\x00")
            xys, ids = np.asarray(im["xys"], float).reshape(-1, 2), np.asarray(im["point3D_ids"], np.int64)
            f.write(struct.pack("<Q", len(ids)))
            rec = np.empty(len(ids), dtype=np.dtype([("x", "<f8"), ("y", "<f8"), ("id", "<i8")]))
            rec["x"], rec["y"], rec["id"] = xys[:, 0], xys[:, 1], ids
            f.write(rec.tobytes())
    with open(os.path.join(out_dir, "points3D.bin"), "wb") as f:
        f.write(struct.pack("<Q", len(model["points"])))
        for pid, p in sorted(model["points"].items()):
            f.write(struct.pack("<Q3d3Bd", pid, *map(float, p["xyz"]), *map(int, p["rgb"]), float(p["error"])))
            tr = np.stack([p["image_ids"], p["point2D_idxs"]], 1).astype("<i4")
            f.write(struct.pack("<Q", len(tr)) + tr.tobytes())


# ------------------------------------------------------------------ text (read only)
def _data_lines(path):
    with open(path) as f:
        return [l.rstrip("\n") for l in f if not l.startswith("#")]


def read_model_txt(d):
    cams, imgs, pts = {}, {}, {}
    for l in _data_lines(os.path.join(d, "cameras.txt")):
        if not l.strip():
            continue
        v = l.split()
        cams[int(v[0])] = dict(model=v[1], width=int(v[2]), height=int(v[3]), params=np.array(v[4:], float))
    lines = _data_lines(os.path.join(d, "images.txt"))   # pairs: header line, observations line (may be empty)
    i = 0
    while i < len(lines):
        if not lines[i].strip():
            i += 1
            continue
        v = lines[i].split()
        obs = np.array(lines[i + 1].split(), float).reshape(-1, 3) if i + 1 < len(lines) else np.zeros((0, 3))
        imgs[int(v[0])] = dict(qvec=np.array(v[1:5], float), tvec=np.array(v[5:8], float), camera_id=int(v[8]),
                               name=" ".join(v[9:]), xys=obs[:, :2], point3D_ids=obs[:, 2].astype(np.int64))
        i += 2
    for l in _data_lines(os.path.join(d, "points3D.txt")):
        if not l.strip():
            continue
        v = l.split()
        tr = np.array(v[8:], int).reshape(-1, 2)
        pts[int(v[0])] = dict(xyz=np.array(v[1:4], float), rgb=np.array(v[4:7], np.uint8), error=float(v[7]),
                              image_ids=tr[:, 0], point2D_idxs=tr[:, 1])
    return dict(cameras=cams, images=imgs, points=pts)


def find_model_dir(scene_dir):
    """gsplat's rule: sparse/0 if it exists, else sparse."""
    for d in (os.path.join(scene_dir, "sparse", "0"), os.path.join(scene_dir, "sparse")):
        if os.path.exists(os.path.join(d, "images.bin")) or os.path.exists(os.path.join(d, "images.txt")):
            return d
    raise FileNotFoundError(f"no COLMAP model under {scene_dir}/sparse")


def read_model(d):
    if os.path.isdir(os.path.join(d, "sparse")):
        d = find_model_dir(d)
    if os.path.exists(os.path.join(d, "images.bin")):
        return dict(cameras=read_cameras_bin(os.path.join(d, "cameras.bin")),
                    images=read_images_bin(os.path.join(d, "images.bin")),
                    points=read_points_bin(os.path.join(d, "points3D.bin")))
    return read_model_txt(d)


# ------------------------------------------------------------------ helpers
def qvec2rotmat(q):
    w, x, y, z = q / np.linalg.norm(q)
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
                     [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]])


def rotmat2qvec(R):
    K = np.array([[R[0, 0] - R[1, 1] - R[2, 2], 0, 0, 0],
                  [R[1, 0] + R[0, 1], R[1, 1] - R[0, 0] - R[2, 2], 0, 0],
                  [R[2, 0] + R[0, 2], R[2, 1] + R[1, 2], R[2, 2] - R[0, 0] - R[1, 1], 0],
                  [R[2, 1] - R[1, 2], R[0, 2] - R[2, 0], R[1, 0] - R[0, 1], R[0, 0] + R[1, 1] + R[2, 2]]]) / 3
    w, V = np.linalg.eigh(K)
    q = V[[3, 0, 1, 2], np.argmax(w)]
    return q if q[0] >= 0 else -q


def poses_by_name(model):
    """{image name: (R (3,3), t (3,))}, world-to-camera."""
    return {im["name"]: (qvec2rotmat(im["qvec"]), np.asarray(im["tvec"], float)) for im in model["images"].values()}


def subset_model(model, keep_names, min_track=2):
    """Keep only images in keep_names; keep 3D points still seen by >= min_track of them.

    gsplat looks every image in the model up in images/ and every track entry up in the model's images,
    so a subset needs both trimmed. Dropped points are set to -1 in the images' observations.
    """
    keep_names = set(keep_names)
    images = {i: dict(im) for i, im in model["images"].items() if im["name"] in keep_names}
    missing = keep_names - {im["name"] for im in images.values()}
    if missing:
        raise KeyError(f"{len(missing)} names not in the model, e.g. {sorted(missing)[:3]}")
    kept_ids = np.array(sorted(images))
    points = {}
    for pid, p in model["points"].items():
        m = np.isin(p["image_ids"], kept_ids)
        if m.sum() >= min_track:
            points[pid] = dict(p, image_ids=p["image_ids"][m], point2D_idxs=p["point2D_idxs"][m])
    for im in images.values():
        ids = np.asarray(im["point3D_ids"], np.int64).copy()
        ids[(ids >= 0) & ~np.isin(ids, np.fromiter(points.keys(), np.int64, len(points)))] = -1
        im["point3D_ids"] = ids
    cams = {c: model["cameras"][c] for c in {im["camera_id"] for im in images.values()}}
    return dict(cameras=cams, images=images, points=points)

In [ ]:
%%writefile /kaggle/tmp/hybrid_sfm/pose_metrics.py
"""Camera pose accuracy against a reference, numpy only.

Metrics (the usual ones in the VGGT / VGGSfM / MASt3R-SfM papers):
- RRA@τ / RTA@τ: share of image pairs whose relative rotation / relative translation direction error is < τ°.
  Translation direction is sign-ambiguous for some methods, so its error is min(e, 180 - e).
- AUC@30: mean over τ = 1..30 of the share of pairs with max(rotation error, translation error) < τ.
- ATE: RMSE of camera centres after a Sim(3) (Umeyama) alignment to the reference, divided by the
  reference's scene scale (median distance of reference centres to their mean), in percent.
Images the method failed to register count as 180° error for every pair they are in, so methods that
drop images are not rewarded for it; the share of registered images is reported too.
"""
import numpy as np


def _angle(R):
    return np.degrees(np.arccos(np.clip((np.trace(R) - 1) / 2, -1.0, 1.0)))


def _vec_angle(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12:
        return 180.0
    return np.degrees(np.arccos(np.clip(a @ b / (na * nb), -1.0, 1.0)))


def umeyama(src, dst):
    """s, R, t minimising ||s R src + t - dst||^2 (src, dst: (N, 3))."""
    mu_s, mu_d = src.mean(0), dst.mean(0)
    xs, xd = src - mu_s, dst - mu_d
    U, S, Vt = np.linalg.svd(xd.T @ xs / len(src))
    D = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        D[2, 2] = -1
    R = U @ D @ Vt
    var_s = (xs ** 2).sum() / len(src)
    s = np.trace(np.diag(S) @ D) / var_s if var_s > 0 else 1.0
    return s, R, mu_d - s * R @ mu_s


def centre(R, t):
    return -R.T @ t


def pose_errors(pred, ref, names=None):
    """pred, ref: {name: (R, t)} world-to-camera. names: the image set to score (default: all of ref)."""
    names = sorted(ref if names is None else names)
    reg = [n for n in names if n in pred]
    out = dict(n_images=len(names), n_registered=len(reg), registered=len(reg) / max(len(names), 1))

    rot_err, tr_err = [], []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            if a not in pred or b not in pred:
                rot_err.append(180.0); tr_err.append(180.0)
                continue
            (Ra, ta), (Rb, tb) = ref[a], ref[b]
            (Pa, pa), (Pb, pb) = pred[a], pred[b]
            R_ref, R_pred = Rb @ Ra.T, Pb @ Pa.T                       # camera a -> camera b
            t_ref, t_pred = tb - R_ref @ ta, pb - R_pred @ pa
            rot_err.append(_angle(R_pred.T @ R_ref))
            e = _vec_angle(t_pred, t_ref)
            tr_err.append(min(e, 180.0 - e))
    rot_err, tr_err = np.array(rot_err), np.array(tr_err)
    worst = np.maximum(rot_err, tr_err)
    for tau in (5, 15):
        out[f"RRA@{tau}"] = float((rot_err < tau).mean()) if len(worst) else float("nan")
        out[f"RTA@{tau}"] = float((tr_err < tau).mean()) if len(worst) else float("nan")
    out["AUC@30"] = float(np.mean([(worst < t).mean() for t in range(1, 31)])) if len(worst) else float("nan")
    out["rot_err_median"] = float(np.median(rot_err)) if len(rot_err) else float("nan")

    if len(reg) >= 3:
        C_ref = np.array([centre(*ref[n]) for n in reg])
        C_pred = np.array([centre(*pred[n]) for n in reg])
        s, R, t = umeyama(C_pred, C_ref)
        res = (s * C_pred @ R.T + t) - C_ref
        scale = np.median(np.linalg.norm(C_ref - C_ref.mean(0), axis=1))
        out["ATE_%"] = float(100 * np.sqrt((res ** 2).sum(1).mean()) / max(scale, 1e-12))
    else:
        out["ATE_%"] = float("nan")
    return out

In [ ]:
%%writefile /kaggle/tmp/hybrid_sfm/vggt_memory.py
"""Peak GPU memory and time of one VGGT pass vs number of images (R1: how big can a chunk be on this GPU?).

Runs the same branches as VGGT's demo_colmap.py (aggregator + camera head + depth head) at 518x518.
Usage (inside the VGGT environment, with the vggt repo on PYTHONPATH):
    python vggt_memory.py <image_dir> <out.json> 8 16 24 32 48 64
Stops at the first size that runs out of memory.
"""
import glob
import json
import sys
import time

import torch
import torch.nn.functional as F


def main():
    image_dir, out = sys.argv[1], sys.argv[2]
    sizes = [int(s) for s in sys.argv[3:]] or [8, 16, 24, 32, 48, 64]
    from vggt.models.vggt import VGGT
    from vggt.utils.load_fn import load_and_preprocess_images
    from vggt.utils.pose_enc import pose_encoding_to_extri_intri

    dev = "cuda"
    dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    model = VGGT()
    model.load_state_dict(torch.hub.load_state_dict_from_url("https://huggingface.co/facebook/VGGT-1B/resolve/main/model.pt"))
    model = model.eval().to(dev)
    weights_gb = torch.cuda.memory_allocated() / 1e9

    paths = sorted(glob.glob(f"{image_dir}/*"))
    rows = []
    for n in sizes:
        chosen = [paths[i % len(paths)] for i in range(n)]
        images = load_and_preprocess_images(chosen, mode="pad").to(dev)
        images = F.interpolate(images, size=(518, 518), mode="bilinear", align_corners=False)
        torch.cuda.synchronize(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        t = time.time()
        try:
            with torch.no_grad():
                with torch.cuda.amp.autocast(dtype=dtype):
                    tokens, ps_idx = model.aggregator(images[None])
                pose_enc = model.camera_head(tokens)[-1]
                pose_encoding_to_extri_intri(pose_enc, images.shape[-2:])
                model.depth_head(tokens, images[None], ps_idx)
            torch.cuda.synchronize()
            row = dict(images=n, peak_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2),
                       seconds=round(time.time() - t, 2), status="ok")
        except torch.cuda.OutOfMemoryError:
            row = dict(images=n, peak_gb=None, seconds=None, status="out of memory")
        print(row, flush=True)
        rows.append(row)
        del images
        torch.cuda.empty_cache()
        if row["status"] != "ok":
            break
    json.dump(dict(gpu=torch.cuda.get_device_name(0), dtype=str(dtype), weights_gb=round(weights_gb, 2),
                   total_gb=round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), rows=rows),
              open(out, "w"), indent=1)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/tmp/hybrid_sfm/run_colmap.py
"""Classical baseline: COLMAP (SIFT + exhaustive matching + incremental mapping) through pycolmap.

One shared PINHOLE camera, so the model matches undistorted inputs (Mip-NeRF 360 images_4 are already
undistorted; a distortion model would make gsplat crop the images and break PSNR comparisons, F1.3).
Usage (VGGT environment, official pycolmap 3.10):
    python run_colmap.py <scene_dir>        # reads <scene_dir>/images, writes <scene_dir>/sparse/0
"""
import json
import os
import shutil
import sys
import time

import pycolmap


def main():
    scene = sys.argv[1]
    images, work = os.path.join(scene, "images"), os.path.join(scene, "colmap_work")
    shutil.rmtree(work, ignore_errors=True)
    os.makedirs(work)
    db = os.path.join(work, "database.db")
    t0 = time.time()
    try:
        pycolmap.extract_features(db, images, camera_mode=pycolmap.CameraMode.SINGLE, camera_model="PINHOLE")
    except TypeError:                                   # older/newer signature: camera model via reader options
        ro = pycolmap.ImageReaderOptions()
        ro.camera_model = "PINHOLE"
        pycolmap.extract_features(db, images, camera_mode=pycolmap.CameraMode.SINGLE, reader_options=ro)
    t1 = time.time()
    pycolmap.match_exhaustive(db)
    t2 = time.time()
    recs = pycolmap.incremental_mapping(db, images, os.path.join(work, "mapping"))
    t3 = time.time()
    if not recs:
        raise SystemExit("COLMAP: no reconstruction")
    best = max(recs.values(), key=lambda r: r.num_reg_images())
    out = os.path.join(scene, "sparse", "0")
    os.makedirs(out, exist_ok=True)
    best.write(out)
    info = dict(models=len(recs), registered=best.num_reg_images(), points=best.num_points3D(),
                extract_s=round(t1 - t0, 1), match_s=round(t2 - t1, 1), map_s=round(t3 - t2, 1),
                total_s=round(t3 - t0, 1), pycolmap=pycolmap.__version__)
    json.dump(info, open(os.path.join(scene, "colmap_info.json"), "w"), indent=1)
    print(info)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/tmp/hybrid_sfm/vggt_colmap_lowmem.py
"""VGGT's demo_colmap.py with less GPU memory, for VGGT + BA on a 16 GB T4.

demo_colmap.py keeps the 1B-parameter model (~5 GB in fp32) on the GPU while it predicts tracks for bundle
adjustment, which runs out of memory at 32 images on a T4. This wrapper moves the model to the CPU as soon as
the forward pass is done. Everything else is VGGT's own code and arguments.
Extra flag: --no_fine_tracking (demo_colmap's --fine_tracking defaults to True and can't be turned off there).
Run from the vggt repo folder (or with it on PYTHONPATH):
    python vggt_colmap_lowmem.py --scene_dir X --use_ba [--max_query_pts 2048 --query_frame_num 5] [--no_fine_tracking]
"""
import sys

import torch

import demo_colmap as dc

_run_vggt = dc.run_VGGT


def _run_then_free(model, *args, **kwargs):
    out = _run_vggt(model, *args, **kwargs)
    model.cpu()
    torch.cuda.empty_cache()
    return out


dc.run_VGGT = _run_then_free

if __name__ == "__main__":
    no_fine = "--no_fine_tracking" in sys.argv
    if no_fine:
        sys.argv.remove("--no_fine_tracking")
    args = dc.parse_args()
    if no_fine:
        args.fine_tracking = False
    with torch.no_grad():
        dc.demo_fn(args)

In [ ]:
open(f'{WORK}/hybrid_sfm/__init__.py', 'a').close()
sys.path.insert(0, WORK)
from hybrid_sfm import colmap_io as cio, pose_metrics as pm
print('hybrid_sfm ready')

## 5. Data: Mip-NeRF 360 scenes (`images_4` + official poses), one at a time

In [ ]:
from remotezip import RemoteZip
import imageio.v2 as imageio, numpy as np
URL = 'http://storage.googleapis.com/gresearch/refraw360/360_v2.zip'
WANT = ('sparse/', 'poses_bounds.npy', 'images_4/')

def get_scene(scene):
    # Downloads only the files that are missing or incomplete, so an interrupted download is finished next time.
    data = f'{WORK}/data/360_v2/{scene}'
    if not os.path.exists(f'{data}/.complete'):
        t = time.time(); n = 0
        with RemoteZip(URL) as z:
            for info in z.infolist():
                m = info.filename
                if not (m.startswith(f'{scene}/') and not m.endswith('/') and m.split('/', 1)[1].startswith(WANT)):
                    continue
                dst = f'{data}/' + m.split('/', 1)[1]
                if os.path.exists(dst) and os.path.getsize(dst) == info.file_size:
                    continue
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                with z.open(m) as src_, open(dst + '.part', 'wb') as f:
                    f.write(src_.read())
                os.replace(dst + '.part', dst); n += 1
        open(f'{data}/.complete', 'w').write('ok')
        print(f'{scene}: downloaded {n} files in {time.time()-t:.0f}s')
    if not os.path.exists(f'{data}/images'):
        os.symlink(f'{data}/images_4', f'{data}/images')      # train the provided 1/4 images with --data_factor 1 (F1.2)
    print(scene, len(os.listdir(f'{data}/images')), 'images | size', imageio.imread(glob.glob(f'{data}/images/*')[0]).shape)
    return data

## 6. How many images fit in one VGGT pass? (memory sweep on a T4)

The same branches as `demo_colmap.py` at 518×518, in fp16, since a T4 has no bf16. This answers question 1, and caps N.

In [ ]:
SWEEP_JSON = f'{OUT}/r1_vggt_memory.json'
if not os.path.exists(SWEEP_JSON):
    imgs = f"{get_scene(SCENES[0])}/images_4"
    r = subprocess.run([VPY, f'{WORK}/hybrid_sfm/vggt_memory.py', imgs, SWEEP_JSON, *map(str, SWEEP)],
                       capture_output=True, text=True, env=VENV_ENV)
    print(r.stdout[-3000:])
    if not os.path.exists(SWEEP_JSON):
        print(r.stderr[-3000:]); raise RuntimeError('VGGT could not run at all: see the error above')
MEM = json.load(open(SWEEP_JSON))
ok = [row for row in MEM['rows'] if row['status'] == 'ok']
fits = [row['images'] for row in ok if row['peak_gb'] <= 0.7 * MEM['total_gb']]   # leave room for BA tracking
N_EFF = min(N_VIEWS, max(fits)) if fits else 8
print(f"{MEM['gpu']} ({MEM['total_gb']} GB, {MEM['dtype']}), weights {MEM['weights_gb']} GB")
for row in MEM['rows']:
    print(f"  {row['images']:>3} images: " + (f"{row['peak_gb']:5.2f} GB peak, {row['seconds']:.1f} s" if row['status'] == 'ok' else row['status']))
print(f'N_EFF = {N_EFF} images per scene' + ('' if N_EFF == N_VIEWS else f' (lowered from {N_VIEWS})'))

## 7. Build the four scene folders per scene

Every method sees the same N images: symlinks to the same files. `gt` gets the official model cut down to those images, with tracks and observations trimmed so gsplat can load it.

In [ ]:
def select_names(names, n):
    names = sorted(names)
    idx = np.unique(np.linspace(0, len(names) - 1, n).round().astype(int))
    return [names[i] for i in idx]

SCENE_SETS = {}
def prepare(scene):
    if scene in OWN_SCENES:
        src_dir, img_dir = OWN_SCENES[scene], f'{OWN_SCENES[scene]}/images'
    else:
        src_dir = get_scene(scene); img_dir = f'{src_dir}/images_4'
    ref = cio.read_model(src_dir)                          # sparse/0 (official / your full COLMAP run)
    in_model = {im['name'] for im in ref['images'].values()}
    names = select_names([n for n in os.listdir(img_dir) if n in in_model], N_EFF)
    for method in METHODS:
        d = f'{R1}/{scene}/{method}'
        os.makedirs(f'{d}/images', exist_ok=True)
        for n in names:
            if not os.path.exists(f'{d}/images/{n}'):
                os.symlink(os.path.abspath(f'{img_dir}/{n}'), f'{d}/images/{n}')
    if not os.path.exists(f'{R1}/{scene}/gt/sparse/0/images.bin'):
        cio.write_model_bin(cio.subset_model(ref, names), f'{R1}/{scene}/gt/sparse/0')
    SCENE_SETS[scene] = dict(names=names, ref=cio.poses_by_name(ref), n_all=len(in_model))
    print(f'{scene}: {len(names)} of {len(in_model)} images | test views (every 8th): {names[::8]}')

for scene in [*SCENES, *OWN_SCENES]:
    prepare(scene)

## 8. Reconstruct: COLMAP, VGGT, VGGT + BA

Each run is timed, and its log is saved. A failure is recorded, and the next run still starts. If VGGT+BA fails at the default settings, it retries with VGGT's lighter tracking settings (`--max_query_pts 2048 --query_frame_num 5`).

In [ ]:
RECON = {}
MIN_POINTS = 1000          # gsplat starts the Gaussians from the model's 3D points; an empty point cloud crashes it
def usable(info, method):
    return info.get('status') == 'ok' and (method == 'colmap' or info.get('points', 0) >= MIN_POINTS)

def reconstruct(scene, method):
    d = f'{R1}/{scene}/{method}'
    info_file = f'{d}/recon_info.json'
    saved = f'{OUT}/r1_models/{scene}/{method}'
    if not os.path.exists(info_file) and os.path.exists(f'{saved}_info.json') and             usable(json.load(open(f'{saved}_info.json')), method):      # finished in an earlier session
        shutil.copytree(saved, f'{d}/sparse', dirs_exist_ok=True); shutil.copy(f'{saved}_info.json', info_file)
    if os.path.exists(info_file) and usable(json.load(open(info_file)), method):
        RECON[(scene, method)] = json.load(open(info_file)); return
    os.makedirs(f'{OUT}/r1_logs', exist_ok=True)
    if method == 'colmap':
        tries = [([VPY, f'{WORK}/hybrid_sfm/run_colmap.py', d], WORK)]
    else:
        base = [VPY, 'demo_colmap.py', '--scene_dir', d]
        lowmem = [VPY, f'{WORK}/hybrid_sfm/vggt_colmap_lowmem.py', '--scene_dir', d, '--use_ba']   # model off the GPU before BA tracking
        light = ['--max_query_pts', '2048', '--query_frame_num', '5']
        # Without BA, demo_colmap keeps only points with depth confidence > 5 (VGGT's conf is >= 1); if that leaves
        # no points (seen on a T4 in fp16), retry with lower thresholds. Poses are the same either way.
        tries = [(base, VGGT_REPO), (base + ['--conf_thres_value', '2'], VGGT_REPO),
                 (base + ['--conf_thres_value', '1.01'], VGGT_REPO)] if method == 'vggt' else \
                [(lowmem, VGGT_REPO), (lowmem + light, VGGT_REPO), (lowmem + light + ['--no_fine_tracking'], VGGT_REPO)]
    for k, (cmd, cwd) in enumerate(tries):
        shutil.rmtree(f'{d}/sparse', ignore_errors=True)
        t = time.time()
        r = subprocess.run(cmd, cwd=cwd, env=VENV_ENV, capture_output=True, text=True)
        open(f'{d}/recon_log_{k}.txt', 'w').write(r.stdout + '\n----- stderr -----\n' + r.stderr)
        shutil.copy(f'{d}/recon_log_{k}.txt', f'{OUT}/r1_logs/{scene}_{method}_{k}.txt')     # survives a reset
        info = dict(method=method, wall_s=round(time.time() - t, 1), args=' '.join(cmd[2:]).replace(d, '<scene>'),
                    status='ok' if r.returncode == 0 else 'failed: ' + (r.stderr.strip().splitlines() or ['?'])[-1][:200])
        if r.returncode == 0 and method != 'colmap':
            try:
                info['points'] = len(cio.read_model(d)['points'])
            except Exception:
                info['points'] = 0
            if info['points'] < MIN_POINTS:
                info['status'] = f"failed: only {info['points']} 3D points"
                print(f"  {scene} {method} attempt {k + 1}: only {info['points']} 3D points, retrying"); continue
        if r.returncode == 0:
            break
        print(f'  {scene} {method} attempt {k + 1} failed. Last lines of the log:\n' + '\n'.join(
            (r.stdout + r.stderr).strip().splitlines()[-25:]))
    if method == 'colmap' and os.path.exists(f'{d}/colmap_info.json'):
        info['colmap'] = json.load(open(f'{d}/colmap_info.json'))
    json.dump(info, open(info_file, 'w'), indent=1)
    RECON[(scene, method)] = info

for scene in SCENE_SETS:
    for method in ['colmap', 'vggt', 'vggt_ba']:
        reconstruct(scene, method)
        print(f"{scene:>8} {method:<8} {RECON[(scene, method)]['status']:<10} {RECON[(scene, method)]['wall_s']:7.1f} s")
    for method in METHODS:                                # keep the small models in the Output (R2 reuses them)
        src = f'{R1}/{scene}/{method}/sparse'
        if os.path.exists(src):
            shutil.copytree(src, f'{OUT}/r1_models/{scene}/{method}', dirs_exist_ok=True)
        if os.path.exists(f'{R1}/{scene}/{method}/recon_info.json'):
            shutil.copy(f'{R1}/{scene}/{method}/recon_info.json', f'{OUT}/r1_models/{scene}/{method}_info.json')

## 9. Pose accuracy against the reference

In [ ]:
POSE = {}
for scene, s in SCENE_SETS.items():
    for method in METHODS:
        d = f'{R1}/{scene}/{method}'
        try:
            pred = cio.poses_by_name(cio.read_model(d))
        except FileNotFoundError:
            POSE[(scene, method)] = None; continue
        POSE[(scene, method)] = pm.pose_errors(pred, s['ref'], s['names'])
    print(f'\n{scene} ({len(s["names"])} images)')
    for method in METHODS:
        m = POSE.get((scene, method))
        print(f'  {method:<8} ' + ('no model' if m is None else
              f"registered {m['n_registered']}/{m['n_images']} | AUC@30 {m['AUC@30']:.3f} | RRA@5 {m['RRA@5']:.3f} | "
              f"RTA@5 {m['RTA@5']:.3f} | ATE {m['ATE_%']:.2f}%"))

## 10. Train the same 3DGS recipe on each set of poses

The trainer and its safeguards are the same as in `02c`. A method with no model, or one that registered too few images, is skipped and marked in the table.

In [ ]:
help_txt = subprocess.run([sys.executable, 'simple_trainer.py', 'default', '--help'], capture_output=True,
                          text=True, cwd=f'{REPO}/examples').stdout
assert re.search(r'lpips.net', help_txt), 'this gsplat has no --lpips_net option'

def step_of(path):
    return int(re.search(r'step(\d+)', os.path.basename(path)).group(1))

def stats_files(rd, stage):
    return sorted(glob.glob(f'{rd}/stats/{stage}_step*.json'), key=step_of)    # numeric order: 6999 before 29999

def check_resolution(rd, data_dir):
    expected = imageio.imread(sorted(glob.glob(f'{data_dir}/images/*'))[0]).shape[1]
    actual = imageio.imread(sorted(glob.glob(f'{rd}/renders/val_step*_*.png'))[0]).shape[1] / 2   # GT | render
    if abs(actual - expected) / expected > 0.1:
        raise RuntimeError(f'Trained at {actual:.0f}px instead of {expected}px: wrong data_factor? See F1.2')
    return int(actual)

def train(scene, exp, args, data):
    rd = f'{PROJECT}/results/{run_name(scene, exp)}'
    done = [f for f in stats_files(rd, 'val') if step_of(f) == STEPS - 1]
    if done:
        print(f'✔ {run_name(scene, exp)}: already trained'); return
    if os.path.exists(rd):                      # a half-finished run can't be resumed: keep its files, start fresh
        shutil.move(rd, rd + time.strftime('_failed_%Y%m%d_%H%M'))
    cmd = [sys.executable, 'simple_trainer.py', *args, '--data_dir', data, '--data_factor', '1',
           '--result_dir', rd, '--max_steps', str(STEPS), '--disable_viewer', '--lpips_net', 'vgg']
    os.makedirs(rd, exist_ok=True)
    print('$', ' '.join(cmd[1:]))
    t0 = last = time.time(); buf = ''; tail = []
    env = {**os.environ, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}   # less memory fragmentation (F2.1)
    p = subprocess.Popen(cmd, cwd=f'{REPO}/examples', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    with open(f'{rd}/train_log.txt', 'w') as lf:
        while (chunk := p.stdout.read(512)):
            lf.write(chunk); buf += chunk
            parts = re.split(r'[\r\n]', buf); buf = parts[-1]
            for ln in filter(str.strip, parts[:-1]):
                tail = (tail + [ln])[-40:]
                if 'PSNR' in ln or time.time() - last > 120:
                    print(f'[{(time.time()-t0)/60:5.1f} min] {ln.strip()[:150]}'); last = time.time()
    if p.wait() != 0:
        print('\n'.join(tail)); raise RuntimeError(f'training failed, log: {rd}/train_log.txt')
    json.dump({'wall_min': round((time.time() - t0) / 60, 1)}, open(f'{rd}/wall_time.json', 'w'))
    print(f'✔ {run_name(scene, exp)} finished in {(time.time()-t0)/60:.1f} min | trained at width {check_resolution(rd, data)}px')


# The fly-through video fits a spline through cameras[5:-5]; with few registered cameras it crashes after the
# eval (scipy "1 <= k <= 5"). R1 doesn't need the video.
if re.search(r'disable.video', globals().get('help_txt', '')) and '--disable_video' not in GSPLAT_ARGS:
    GSPLAT_ARGS = GSPLAT_ARGS + ['--disable_video']
STATUS = {}
for scene in SCENE_SETS:
    for method in METHODS:
        m = POSE.get((scene, method))
        if m is None or m['n_registered'] < 8:
            why = (RECON.get((scene, method), {}).get('status', 'no model') if m is None
                   else f"only {m['n_registered']} images registered")
            STATUS[(scene, method)] = f'skipped: {why}'; continue
        try:
            train(scene, method, GSPLAT_ARGS, f'{R1}/{scene}/{method}'); STATUS[(scene, method)] = 'finished'
        except Exception as e:
            STATUS[(scene, method)] = f'failed: {str(e)[:200]}'; print(f'❌ {scene} {method}: {e}')
        import gc; gc.collect()
print(STATUS)

## 11. Results (`r1_results.md` / `.json` in the Output)

In [ ]:
import torch
def last_val(rd):
    vals = {step_of(f): json.load(open(f)) for f in stats_files(rd, 'val')}
    trains = {step_of(f): json.load(open(f)) for f in stats_files(rd, 'train')}
    if STEPS - 1 not in vals: return None, None
    return vals[STEPS - 1], trains.get(STEPS - 1, {})

fmt = lambda v, f='{:.3f}': '–' if v is None or (isinstance(v, float) and v != v) else f.format(v)
lines = [f'## R1: VGGT vs VGGT+BA vs COLMAP ({torch.cuda.get_device_name(0)}, VGGT commit {VGGT_COMMIT})', '',
         f'**VGGT memory, one pass at 518×518 ({MEM["dtype"]}), weights {MEM["weights_gb"]} GB:**', '',
         '| Images | Peak GPU memory | Time |', '|---|---|---|']
for row in MEM['rows']:
    lines.append(f"| {row['images']} | " + (f"{row['peak_gb']:.2f} GB | {row['seconds']:.1f} s |" if row['status'] == 'ok' else f"{row['status']} | – |"))
RESULTS = dict(gpu=torch.cuda.get_device_name(0), vggt_commit=VGGT_COMMIT, n_views=N_EFF, steps=STEPS, cap=CAP,
               memory=MEM, scenes={})
for scene, s in SCENE_SETS.items():
    lines += ['', f'**{scene}**: {len(s["names"])} of {s["n_all"]} images, 3DGS = mcmc {CAP // 1000}k, {STEPS // 1000}k steps, test = every 8th image', '',
              '| Method | Registered | Pose time | AUC@30 | RRA@5 | RTA@5 | ATE | #Gaussians | PSNR | ΔPSNR vs gt | SSIM | LPIPS (vgg) | 3DGS time |',
              '|---|---|---|---|---|---|---|---|---|---|---|---|---|']
    gt_v, _ = last_val(f'{PROJECT}/results/{run_name(scene, "gt")}')
    RESULTS['scenes'][scene] = {}
    for method in METHODS:
        m, rec = POSE.get((scene, method)), RECON.get((scene, method), {})
        v, tr = last_val(f'{PROJECT}/results/{run_name(scene, method)}')
        RESULTS['scenes'][scene][method] = dict(pose=m, recon=rec, val=v, train=tr, status=STATUS.get((scene, method)))
        reg = '–' if m is None else f"{m['n_registered']}/{m['n_images']}"
        ptime = '(reference)' if method == 'gt' else (f"{rec.get('wall_s', float('nan')):.0f} s" if rec.get('status') == 'ok' else f"*{rec.get('status', 'not run')}*")
        pose_cells = ['–'] * 4 if m is None else [fmt(m['AUC@30']), fmt(m['RRA@5']), fmt(m['RTA@5']), fmt(m['ATE_%'], '{:.2f}%')]
        if v is None or STATUS.get((scene, method)) != 'finished':     # only scores from a run that finished cleanly
            gs_cells = ['–', f"*{STATUS.get((scene, method), 'not run')}*", '–', '–', '–', '–']
        else:
            d = '–' if gt_v is None else f"{v['psnr'] - gt_v['psnr']:+.2f} dB"
            gs_cells = [f"{v['num_GS']:,}", f"{v['psnr']:.2f}", d, f"{v['ssim']:.3f}", f"{v['lpips']:.3f}",
                        f"{tr.get('ellipse_time', float('nan')) / 60:.1f} min"]
        lines.append(f"| {method} | {reg} | {ptime} | " + ' | '.join(pose_cells + gs_cells) + ' |')
lines += ['', 'Pose time = wall time of the whole step (VGGT: includes loading the 5 GB weights and the images; COLMAP: CPU). '
          'Pose metrics are against the official poses (COLMAP on all images) for the same images; unregistered images count as 180° error.']
table = '\n'.join(lines)
open(f'{OUT}/r1_results.md', 'w', encoding='utf-8').write(table + '\n')
json.dump(RESULTS, open(f'{OUT}/r1_results.json', 'w'), indent=1, default=float)
print(table)

## 12. One test view per method

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
for scene in SCENE_SETS:
    files = {m: f'{PROJECT}/results/{run_name(scene, m)}/renders/val_step{STEPS - 1}_0000.png' for m in METHODS}
    files = {m: f for m, f in files.items() if os.path.exists(f)}
    if not files: continue
    first = imageio.imread(next(iter(files.values())))[..., :3]
    w = first.shape[1] // 2
    fig, ax = plt.subplots(1, len(files) + 1, figsize=(5 * (len(files) + 1), 4))
    ax[0].imshow(first[:, :w]); ax[0].set_title('real (test view 0)')
    for a, (m, f) in zip(ax[1:], files.items()):
        im = imageio.imread(f)[..., :3].astype(np.float32) / 255
        gt, pr = im[:, :w], im[:, w:]
        a.imshow(pr); a.set_title(f'{m}: PSNR {-10 * np.log10(((gt - pr) ** 2).mean()):.2f}')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.savefig(f'{FIG}/r1_{scene}_views.jpg', dpi=80); plt.close()
    print('saved', f'{FIG}/r1_{scene}_views.jpg')

for f in glob.glob(f'{PROJECT}/results/r1_*/ckpts/*.pt'):     # keep the Output small
    os.remove(f)
print(subprocess.run(['du', '-sh', PROJECT], capture_output=True, text=True).stdout)

## ✅ Checklist
- [ ] Paste `r1_results.md` from the Output into the chat (the README and proposal get the numbers)
- [ ] Note the largest VGGT chunk that fit: R2's chunk size
- [ ] Optional: download `figures/r1_*_views.jpg`